Tutorial 6: Fitting
===================

In the previous tutorial, we chose a lens and source, predicted their lensed
image and simulated a telescope observation. We will now work in the other
direction: starting with an image, we want to learn about the objects that
produced it.

A lens model is a mathematical description of the lens and source. Its
parameters specify properties such as the lens's mass distribution and the
source's position, size and brightness.

To assess a model, we calculate the image it predicts and compare that
prediction with the data. We account for the telescope's blurring and the
uncertainty in each pixel, so that differences are judged relative to the
noise.

We will first evaluate a model using the parameters that generated our
simulated data. We will then change those parameters and inspect how the
agreement changes. Finally, you will try adjusting parameters yourself.

This introduces model fitting: finding parameter values that describe the
data well. Creating a `FitImaging` object evaluates the parameters we supply;
it does not search for new values automatically.

__Contents__

- **Dataset:** Locate the imaging data saved in Tutorial 5.
- **Dataset Auto-Simulation:** Create the dataset if needed, then load it.
- **Mask:** Select the region of the image used in the comparison.
- **Masked Grid:** Inspect the coordinates of the selected pixels.
- **Fitting:** Compare a predicted image with the data and calculate fit statistics.
- **Incorrect Fit:** See how changing the source position affects the comparison.
- **Model Fitting:** Experiment with adjusting parameters yourself.
- **Likelihood Function:** Combine the calculation into a function of the model parameters.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/HowToLens/blob/main/notebooks/chapter_1_introduction/tutorial_1_grids_and_galaxies.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("howtolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

import numpy as np
from pathlib import Path
import autolens as al
import autolens.plot as aplt

__Dataset__

We will use the simulated imaging dataset saved in Tutorial 5. It contains
the background-subtracted image, its noise map and the PSF.

The files are in `dataset/imaging/howtolens/`, relative to the repository root
from which this script is run.

In [ ]:
dataset_path = Path("dataset") / "imaging" / "howtolens"

__Dataset Auto-Simulation__

If the dataset is missing, the following code runs Tutorial 5 to create it.
If you have already run that tutorial, we load the saved files.

The pixel scale is 0.1 arcseconds, matching the simulation.

In [ ]:
al.util.dataset.simulate_if_needed(
    dataset_path, simulator="scripts/chapter_1_introduction/tutorial_5_data.py"
)

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    psf_path=dataset_path / "psf.fits",
    pixel_scales=0.1,
)

Recall the three components of an imaging dataset:

- `data`: the measured brightness in each pixel.
- `noise_map`: the estimated noise standard deviation in each pixel.
- `psf`: the kernel describing how the telescope spreads light.

The data and noise map have the same units. The PSF kernel is normalised
to sum to one.

Below, we print the first value in each array and display the dataset.
The first image pixel and first noise-map pixel refer to the same sky
position; the first PSF value belongs to the separate, smaller kernel.

In [ ]:
print("Value of first pixel in imaging data:")
print(dataset.data.native[0, 0])
print("Value of first pixel in noise map:")
print(dataset.noise_map.native[0, 0])
print("Value of first pixel in PSF:")
print(dataset.psf.kernel.native[0, 0])

aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

A mask selects which pixels we include when comparing the model with the data.
It can exclude unrelated objects, detector defects or regions outside the
area we want to analyse.

For this example, we keep a circular region with a radius of 3 arcseconds,
centred on the lens. This includes the lensed arcs and some surrounding sky.

We do not select individual pixels according to their measured
signal-to-noise ratio. Faint pixels still contain useful information: a model
that predicts bright emission where little is observed should disagree with
those data. The noise map determines how strongly each pixel contributes to
the comparison.

In a `Mask2D`, `True` means excluded and `False` means included.

In [ ]:
mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=3.0,  # Arcseconds.
    centre=(0.0, 0.0),
)

print(mask)
print(mask[48:53, 48:53])

Before applying the mask, let's display its boundary over the full image.
The boundary marks the edge of the region we will include in the comparison.

Check that the lensed arcs lie inside it and that some surrounding sky is
included. At this stage, we are only displaying the boundary; the dataset
has not yet been masked.

In [ ]:
aplt.plot_array(
    array=dataset.data,
    mask=mask.derive_grid.edge,
    title="Imaging Data With Mask Boundary",
)

We now apply the mask to the dataset. Pixels outside the circular region
will be excluded when we compare the model with the data.

In [ ]:
dataset = dataset.apply_mask(mask=mask)

The plotted image now shows the selected region, with the view zoomed to
the unmasked area. The plotting function obtains the mask boundary from
the array itself.

In [ ]:
aplt.plot_array(array=dataset.data, title="Masked Imaging Data")

The mask is stored with the dataset, so later calculations can identify
which pixels to include.

In [ ]:
print("Mask2D:")
print(dataset.mask)

The masked data retain the `native` and `slim` representations introduced
earlier:

- `native` arranges the values in the original 2D image shape. Masked pixels
  are filled with zeros.
- `slim` contains only the unmasked pixel values in a 1D array.

Those zeros in the native representation are placeholders. They do not mean
that zero light was measured outside the mask, and they do not contribute
to the comparison with the model.

Our original image has 101 x 101 pixels. The slim array is shorter because
we have selected only the circular region.

In [ ]:
print("Native image shape:")
print(dataset.data.native.shape)

print("Slim image shape:")
print(dataset.data.slim.shape)

Despite its name, `pixels_in_mask` counts the unmasked pixels: those included
in the analysis. This number should match the length of the slim array.

In [ ]:
print("Number of unmasked pixels:")
print(dataset.mask.pixels_in_mask)

The image and noise map share the same mask and pixel ordering. Their first
slim entries therefore describe the measurement and uncertainty at the same
sky position.

In [ ]:
print("First unmasked image value:")
print(dataset.data.slim[0])

print("First unmasked noise map value:")
print(dataset.noise_map.slim[0])

In the native representation, the corner pixel is masked and therefore
contains a zero placeholder. The central pixel is unmasked and retains its
measured value.

In [ ]:
print("Masked corner pixel in the native image:")
print(dataset.data.native[0, 0])

centre = tuple(s // 2 for s in dataset.data.shape_native)

print("Unmasked central pixel in the native image:")
print(dataset.data.native[centre])

__Masked Grid__

To predict the image, we need the sky coordinates corresponding to the
selected pixels. We obtain these from the mask as a grid of `(y, x)`
coordinates.

Each coordinate marks the centre of an unmasked pixel. Plotting the grid
shows the circular region that we selected.

In [ ]:
masked_grid = mask.derive_grid.unmasked

aplt.plot_grid(grid=masked_grid, title="Masked Grid2D")

We can overlay these coordinates on the image to see where the selected
pixels lie. Each dot corresponds to one pixel whose measured brightness
will be compared with the model prediction.

The grid, image and noise map use matching pixel ordering, so their slim
entries refer to the same positions.

In [ ]:
aplt.plot_array(
    array=dataset.data,
    grid=masked_grid,
    title="Image Data With 2D Grid Overlaid",
)

__Fitting__

We will begin with the same lens and source parameters used to simulate the
data. This gives us a useful reference: we know the underlying model, so we
can see what agreement with a noisy observation should look like.

We should not expect the predicted image to match every observed pixel
exactly. The data contain random noise, whereas our prediction describes
the expected brightness.

As before, the lens has an Isothermal mass profile and the source has a
Sersic light profile. The lens galaxy's light was omitted from this simulation.

In [ ]:
lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        einstein_radius=1.6,
        ell_comps=(0.17647, 0.0),
    ),
)

source_galaxy = al.Galaxy(
    redshift=1.0,
    bulge=al.lp.Sersic(
        centre=(0.1, 0.1),
        ell_comps=(0.0, 0.111111),
        intensity=1.0,
        effective_radius=1.0,
        sersic_index=2.5,
    ),
)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

First, we evaluate the tracer on the dataset's grid. This calculates the
lensed source image at the selected image-plane coordinates.

The arcs should look familiar, but this image has not yet been blurred by
the telescope's PSF. We must include that blurring before comparing it with
the observed data.

In [ ]:
aplt.plot_array(
    array=tracer.image_2d_from(grid=dataset.grid),
    title="Lensed Image Before PSF Blurring",
)

The `FitImaging` object brings together our dataset and tracer. It calculates
the predicted image with PSF blurring and compares it with the data.

We supply the parameter values through the tracer. Creating this object
evaluates that choice of parameters; it does not adjust them.

We will inspect the predicted image first, then calculate the differences
between it and the data. These differences will form the basis of the
statistical quantities introduced below.

In [ ]:
fit = al.FitImaging(dataset=dataset, tracer=tracer)

The fit's `model_data` is the predicted image after PSF convolution. It has
the same units as the observed data and can be compared with them pixel
by pixel.

The mask selects which observed pixels enter the comparison, but light
outside its boundary can still be blurred into those pixels by the PSF.
`FitImaging` therefore also evaluates the surrounding light needed for
this convolution.

This is the same principle as the padding in Tutorial 5: we must account
for light beyond the edge of the region whose blurred image we want.

Below, compare the data and model image. Their arcs should have similar
positions, shapes and brightnesses. The model image is smooth, while the
data include noise. We do not add a new random noise realisation to the
model image when assessing the fit.

In [ ]:
print("First model image pixel:")
print(fit.model_data.slim[0])

aplt.plot_array(array=fit.data, title="Data")
aplt.plot_array(array=fit.model_data, title="Model Image")

Subtracting the model image from the data gives the `residual_map`:

\[
\text{residual} = \text{data} - \text{model\_data}
\]

A positive residual means that the observed pixel is brighter than predicted.
A negative residual means that it is fainter.

We calculate this difference explicitly below and compare its first value
with the value provided by `FitImaging`.

In [ ]:
residual_map = dataset.data - fit.model_data

print("First residual-map pixel:")
print(residual_map.slim[0])

print("First residual-map pixel via fit:")
print(fit.residual_map.slim[0])

aplt.plot_array(array=fit.residual_map, title="Residual Map")

Because we used the parameters that generated the simulation, we expect
the residuals to be dominated by the noise in the data.

Look for positive and negative fluctuations rather than coherent arc-shaped
features. The fluctuations can be larger in bright regions, where the
Poisson noise is larger.

To judge whether a residual is large or small, we must compare it with
the noise expected in that pixel. This is our next step.

**🤖 Ask AI:** Why can a correct lens model leave larger absolute residuals
in bright arcs than in the faint background?

The size of a residual only becomes informative when compared with the
uncertainty in that pixel. A difference of 0.1 electrons per second could
be small in a noisy pixel but substantial in a precisely measured one.

The `normalized_residual_map` divides each residual by its noise standard
deviation:

\[
r_i = \frac{d_i - m_i}{\sigma_i}
\]

Here, \(d_i\) is the observed value, \(m_i\) is the model prediction and
\(\sigma_i\) is the noise-map value for pixel \(i\).

A normalised residual of +2 means that the observation is two noise standard
deviations above the prediction. A value of -2 means that it is two standard
deviations below.

For a suitable model and well-estimated noise, we expect fluctuations around
zero, typically of order one. We do not expect every pixel to be close to zero.

In [ ]:
normalized_residual_map = residual_map / dataset.noise_map

print("First normalized residual-map pixel:")
print(normalized_residual_map.slim[0])

print("First normalized residual-map pixel via fit:")
print(fit.normalized_residual_map.slim[0])

aplt.plot_array(
    array=fit.normalized_residual_map,
    title="Normalized Residual Map",
)

Squaring the normalised residuals gives the `chi_squared_map`:

\[
\chi_i^2 = \left(\frac{d_i - m_i}{\sigma_i}\right)^2
\]

Each value measures one pixel's contribution to the overall disagreement.
The values are non-negative: residuals of +2 and -2 both contribute 4.

Look for pixels with large contributions and whether they form a coherent
pattern. A few large values can occur through noise, especially when an image
contains many pixels.

For our reference model, the contributions should average approximately one,
rather than all being close to zero.

In [ ]:
chi_squared_map = normalized_residual_map**2

print("First chi-squared pixel:")
print(chi_squared_map.slim[0])

print("First chi-squared pixel via fit:")
print(fit.chi_squared_map.slim[0])

aplt.plot_array(array=fit.chi_squared_map, title="Chi Squared Map")

We sum the contributions from all \(N\) unmasked pixels to obtain one number,
the total `chi_squared`:

\[
\chi^2 = \sum_{i=1}^{N}
\left(\frac{d_i - m_i}{\sigma_i}\right)^2
\]

Squaring before summing prevents positive and negative residuals from
cancelling. Large differences relative to the noise contribute strongly.

When comparing parameter choices using the same data, mask and noise map,
a lower chi-squared means smaller overall noise-weighted differences.

In [ ]:
chi_squared = np.sum(chi_squared_map.slim)

print("Chi-squared = ", chi_squared)
print("Chi-squared via fit = ", fit.chi_squared)

Dividing the total chi-squared by the number of unmasked pixels gives the
average contribution per pixel:

\[
\frac{\chi^2}{N}
\]

For the parameters that generated our simulation, we expect this to be
approximately one if the noise estimates describe the fluctuations well.
Individual pixels can contribute much more or much less than one.

A substantially larger value can indicate a mismatch between the model and
data, underestimated noise, or both. A substantially smaller value can
indicate overestimated noise; it does not automatically establish overfitting.

You may also encounter the term "reduced chi-squared". This divides chi-squared
by the number of degrees of freedom, often \(N-p\) when \(p\) independent
parameters have been estimated from the data. Here, we are simply inspecting
the average per pixel for a specified model.

In [ ]:
chi_squared_per_pixel = chi_squared / dataset.mask.pixels_in_mask

print("Chi-squared per pixel = ", chi_squared_per_pixel)

To turn this comparison into a likelihood, we need a statistical description
of the noise.

We will treat each pixel's measurement as an independent Gaussian random
variable, centred on the model prediction \(m_i\), with standard deviation
\(\sigma_i\). This is an approximation to the Poisson counts used in our
simulation, appropriate when the expected counts are sufficiently large.

The probability density for a pixel value is then:

\[
p(d_i \mid m_i, \sigma_i)
=
\frac{1}{\sqrt{2\pi\sigma_i^2}}
\exp\left[-\frac{1}{2}
\left(\frac{d_i-m_i}{\sigma_i}\right)^2\right]
\]

The exponential penalises predictions far from the observed value. The factor
in front normalises the Gaussian so that its density integrates to one.

For independent pixels, we multiply these densities. Taking the natural
logarithm turns that product into a sum:

\[
\log \mathcal{L}
=
-\frac{1}{2}
\left[
\chi^2 + \sum_{i=1}^{N}\log(2\pi\sigma_i^2)
\right]
\]

The second term is called `noise_normalization` in PyAutoLens. It is part of
the probability density, not a test of whether the noise is Gaussian.

For our fixed dataset, mask and noise map, this term stays constant as we
change the lens and source parameters.

In [ ]:
noise_normalization = np.sum(np.log(2 * np.pi * dataset.noise_map.slim**2))

print("Noise Normalization = ", noise_normalization)
print("Noise Normalization via fit = ", fit.noise_normalization)

We can now calculate the log likelihood and compare it with the value
returned by `FitImaging`.

The factor of -0.5 follows from the Gaussian expression above. With the
noise normalisation fixed, reducing chi-squared increases the log likelihood.

In [ ]:
log_likelihood = -0.5 * (chi_squared + noise_normalization)

print("Log Likelihood = ", log_likelihood)
print("Log Likelihood via fit = ", fit.log_likelihood)

For the same dataset, mask and noise assumptions, a higher log likelihood
indicates better agreement between the predicted image and the data.

For example, a log likelihood of 5310 is higher than 5300, so the former
parameter choice provides the better fit under those assumptions.

The absolute value is not a probability that the model is correct. Nor does
a difference in log likelihood have a universal conversion into a number
of standard deviations.

We should also inspect where the model disagrees with the data. The subplot
below brings together the data, model image and residual diagnostics, along
with views of the source.

Look for residuals consistent with noise rather than coherent features
tracing the lensed arcs.

In [ ]:
aplt.subplot_fit_imaging(fit=fit)

__Incorrect Fit__

We will now change the source position while keeping the lens mass and all
other source properties fixed.

The original source centre was `(0.1, 0.1)` arcseconds. We move it to
`(0.15, 0.15)`, a displacement of 0.05 arcseconds in each source-plane
coordinate.

Although 0.05 arcseconds is half our image pixel scale, lensing maps this
source-plane displacement into changes in the positions and shapes of the
observed arcs. It does not simply move every image by half a pixel.

The original `fit` remains our reference comparison.

In [ ]:
source_galaxy = al.Galaxy(
    redshift=1.0,
    bulge=al.lp.Sersic(
        centre=(0.15, 0.15),
        ell_comps=(0.0, 0.111111),
        intensity=1.0,
        effective_radius=1.0,
        sersic_index=2.5,
    ),
)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

aplt.plot_array(
    array=tracer.image_2d_from(grid=dataset.grid),
    title="Lensed Image With Shifted Source",
)

We evaluate this new tracer against the same dataset and inspect the fit.

Look for coherent positive and negative residuals around the arcs. These
indicate regions where the shifted source predicts too little or too much
light, rather than just random fluctuations about the observation.

In [ ]:
fit_bad = al.FitImaging(dataset=dataset, tracer=tracer)

aplt.subplot_fit_imaging(fit=fit_bad)

Compare the log likelihood with our reference value. We expect the shifted
source to give a lower value because its predicted image disagrees with
the data more strongly.

In [ ]:
print("Reference log likelihood:")
print(fit.log_likelihood)

print("Shifted-source log likelihood:")
print(fit_bad.log_likelihood)

print("Change from reference:")
print(fit_bad.log_likelihood - fit.log_likelihood)

Next, move the source centre to `(0.5, 0.5)` arcseconds. All other parameters
remain unchanged.

This is a much larger displacement from the original `(0.1, 0.1)` position.
Compare the predicted arcs and residuals with the previous two examples.

In [ ]:
source_galaxy = al.Galaxy(
    redshift=1.0,
    bulge=al.lp.Sersic(
        centre=(0.5, 0.5),
        ell_comps=(0.0, 0.111111),
        intensity=1.0,
        effective_radius=1.0,
        sersic_index=2.5,
    ),
)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

fit_very_bad = al.FitImaging(dataset=dataset, tracer=tracer)

aplt.subplot_fit_imaging(fit=fit_very_bad)

Print the three log likelihoods together and compare their ordering with
what you saw in the residual plots.

For this example, the larger source displacement should produce the strongest
disagreement and the lowest log likelihood.

In [ ]:
print("Reference source centre (0.1, 0.1):")
print(fit.log_likelihood)

print("Source centre (0.15, 0.15):")
print(fit_bad.log_likelihood)

print("Source centre (0.5, 0.5):")
print(fit_very_bad.log_likelihood)

We have evaluated three choices of parameters, rather than searched for
the best-fitting values.

The parameters used to generate the simulation provide a useful reference,
but they need not give the highest possible likelihood for one noisy
observation. Small parameter changes can sometimes improve agreement with
that particular noise realisation.

To restore the original comparison, rerun the first tracer and `FitImaging`
cells in the Fitting section.

**🤖 Ask AI:** Why might the parameters that maximise the likelihood of a
noisy simulated image differ slightly from those used to generate it?

__Model Fitting__

So far, we have compared parameter choices with a known reference model.
For a real observation, we do not know the lens and source parameters in
advance. Finding values that describe the data well is the task of model
fitting.

Let's try this manually using a new simulated dataset. It contains an
`IsothermalSph` lens mass profile and an `ExponentialCoreSph` source light
profile, with no light from the lens galaxy.

For this exercise, treat the parameters used to generate the data as unknown.
We load the image and apply a circular mask as before.

In [ ]:
dataset_name = "simple__no_lens_light__mass_sis"
dataset_path = Path("dataset") / "imaging" / dataset_name

al.util.dataset.simulate_if_needed(
    dataset_path, simulator="scripts/simulator/no_lens_light__mass_sis.py"
)

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=3.0,
)

dataset = dataset.apply_mask(mask=mask)

aplt.subplot_imaging_dataset(dataset=dataset)

The parameters below are a starting guess. Run the cell and inspect the
predicted image, residuals and log likelihood.

The lens and source centres are `(y, x)` coordinates in arcseconds.
The Einstein radius, effective radius and break radius are also in
arcseconds. The source intensity uses the image's units of electrons
per second per pixel.

In [ ]:
lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.IsothermalSph(
        centre=(1.0, 1.0),
        einstein_radius=1.0,
    ),
)

source_galaxy = al.Galaxy(
    redshift=1.0,
    bulge=al.lp.ExponentialCoreSph(
        centre=(1.0, 1.0),
        intensity=1.0,
        effective_radius=1.0,
        radius_break=0.025,
    ),
)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

fit = al.FitImaging(dataset=dataset, tracer=tracer)

aplt.subplot_fit_imaging(fit=fit)

print("Log Likelihood:")
print(fit.log_likelihood)

Try changing one parameter at a time and rerunning the cell above. Record
the log likelihood for each choice and use the residual plots to understand
what changed.

Start with the lens centre. Try `(0.5, 0.5)` and `(0.0, 0.0)` instead of
`(1.0, 1.0)`. This changes the location about which the lens deflects light.

Next, try changing the source centre in the same way. This changes how the
source is positioned behind the lens and therefore the arrangement of its
multiple images.

You can then explore:

- `einstein_radius`: try 1.3 and 1.6 arcseconds instead of 1.0. This changes
  the characteristic angular scale of the lensing.
- `effective_radius`: try 0.5 and 0.2 arcseconds instead of 1.0. This changes
  the source size and the extent of its lensed features.
- `intensity`: try 0.5 and 0.2 instead of 1.0. With the other parameters
  fixed, this scales the predicted brightness.

Keep `radius_break=0.025` and both redshifts fixed for this exercise.

Retain changes that improve the log likelihood, but continue inspecting
the residuals. The parameters interact, so a useful change at one stage
may need revisiting after another parameter changes.

To restore the original starting point, set both centres to `(1.0, 1.0)`,
and set `einstein_radius`, `effective_radius` and `intensity` to 1.0, then
rerun the cell.

You have now performed a simple manual search: choose parameters, predict
an image, compare it with the data, then use that comparison to choose
another set of parameters.

The aim is not to make every residual vanish. A suitable prediction should
leave fluctuations consistent with the noise, without substantial coherent
structure.

Nor does finding a good-looking result establish that it is the best possible
solution or tell us the uncertainty in its parameters. Several parameter
combinations may describe the data similarly well.

Manual exploration becomes difficult as the number of parameters grows.
Chapter 2 introduces methods that explore parameter choices systematically
and quantify what we can infer from the data.

**🤖 Ask AI:** How could changing both the source size and its intensity
make it difficult to determine either parameter by adjusting them manually?

__Likelihood Function__

Everything we have learned can now be put into a single sequence:

1. Choose the lens and source parameters.
2. Construct their profiles and ray-trace the source to predict a lensed image.
3. Convolve that image with the telescope's PSF.
4. Subtract the prediction from the observed data.
5. Divide the residuals by the noise standard deviations and sum their squares.
6. Include the Gaussian normalisation to calculate the log likelihood.

For a fixed dataset, this sequence defines a function of the model parameters:
the likelihood function. We usually work with its natural logarithm, the
log-likelihood function.

Writing the parameters collectively as \(\boldsymbol{\theta}\), we have:

\[
\log \mathcal{L}(\boldsymbol{\theta})
=
-\frac{1}{2}
\sum_{i=1}^{N}
\left[
\frac{(d_i-m_i(\boldsymbol{\theta}))^2}{\sigma_i^2}
+
\log(2\pi\sigma_i^2)
\right]
\]

The data, mask, PSF and noise map remain fixed. Changing the parameters changes
the predicted image \(m_i(\boldsymbol{\theta})\), and therefore the value
returned by the function.

The function below implements this sequence for our exercise. `FitImaging`
handles the image prediction and PSF convolution, including light that blurs
across the mask boundary. We then write out the statistical calculation
explicitly using the unmasked pixels.

The redshifts and source break radius remain fixed, as in the exercise.

In [ ]:


def log_likelihood_function(
    dataset,
    lens_centre,
    einstein_radius,
    source_centre,
    intensity,
    effective_radius,
):
    lens_galaxy = al.Galaxy(
        redshift=0.5,
        mass=al.mp.IsothermalSph(
            centre=lens_centre,
            einstein_radius=einstein_radius,
        ),
    )

    source_galaxy = al.Galaxy(
        redshift=1.0,
        bulge=al.lp.ExponentialCoreSph(
            centre=source_centre,
            intensity=intensity,
            effective_radius=effective_radius,
            radius_break=0.025,
        ),
    )

    tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])
    fit = al.FitImaging(dataset=dataset, tracer=tracer)

    model_image = fit.model_data.slim
    residuals = dataset.data.slim - model_image
    normalized_residuals = residuals / dataset.noise_map.slim

    chi_squared = np.sum(normalized_residuals**2)
    noise_normalization = np.sum(np.log(2 * np.pi * dataset.noise_map.slim**2))

    return -0.5 * (chi_squared + noise_normalization)


Evaluate the function using your current exercise parameters. Its result
should agree with the log likelihood from the corresponding `FitImaging`
object.

In [ ]:
exercise_log_likelihood = log_likelihood_function(
    dataset=dataset,
    lens_centre=lens_galaxy.mass.centre,
    einstein_radius=lens_galaxy.mass.einstein_radius,
    source_centre=source_galaxy.bulge.centre,
    intensity=source_galaxy.bulge.intensity,
    effective_radius=source_galaxy.bulge.effective_radius,
)

print("Log likelihood from our function:")
print(exercise_log_likelihood)

print("Log likelihood from FitImaging:")
print(fit.log_likelihood)

Each call evaluates one choice of parameters and returns one number. It does
not search for better parameters, and it does not generate new random noise.

The notion of a likelihood function is central to Chapter 2. The search
methods introduced there repeatedly supply different parameter choices to
this function and use the returned likelihoods to explore which choices
describe the data well. Understanding this sequence is therefore the
foundation for understanding how automated lens modelling works.

The interferometer, point-source, weak-lensing, group-scale and cluster-scale
examples shown in Tutorial 5 each have their own bespoke likelihood
functions, tailored to their measurements and analysis. They follow the
same principle: predict the observables from a set of parameters, then compare
those predictions with the data using the appropriate uncertainties.

The `autolens_workspace` provides step-by-step guides to every likelihood
function in PyAutoLens. These unpack the calculations for different dataset
types and analysis methods, just as we have done here.

For the imaging calculation introduced in this tutorial, see the workspace's
[imaging likelihood-function guide](https://github.com/PyAutoLabs/autolens_workspace/blob/main/scripts/imaging/likelihood_function.py).
The corresponding sections of the workspace provide guides for the other
likelihood functions.

You have now followed the full path from a lens and source description to a
quantitative comparison with an observation. In Chapter 2, we will use this
likelihood function to move from trying parameters by hand to exploring them
systematically.